# 📄 Document Image Classification using Lightweight CNNs
## with OCR-based Deep Learning Baseline

**Project:** Compare CNN image classification vs OCR + PyTorch MLP text classification  
**Framework:** PyTorch (all deep learning models)  
**Platform:** Windows / Local

---

### 🗺️ Notebook Map
| Section | Description |
|---------|-------------|
| 1 | Setup & Imports |
| 2 | Configuration — **set DATA_DIR here** |
| 3 | Dataset Loading (ImageFolder) |
| 4 | CNN Pipeline (MobileNetV2) |
| 5 | OCR Text Extraction |
| 6 | OCR-MLP Pipeline (PyTorch) |
| 7 | Evaluation & Comparison |
| 8 | Robustness Testing |
| 9 | Fusion Model |

---
## 🔧 Section 1 — Setup & Imports

**Before running:**
1. Install Tesseract OCR (Windows): https://github.com/UB-Mannheim/tesseract/wiki
2. Set `tesseract_cmd` path in Section 2 if needed
3. Run `pip install torch torchvision pytesseract opencv-python scikit-learn tqdm matplotlib seaborn pillow` in your terminal

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os, sys, re, json, pickle, random, shutil, tempfile, warnings
from pathlib import Path
from collections import defaultdict

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

# ── OpenCV ────────────────────────────────────────────────────────────────────
import cv2

# ── OCR ───────────────────────────────────────────────────────────────────────
import pytesseract

# ── Sklearn (TF-IDF only) ─────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import datasets, transforms, models
from torchvision.models import MobileNet_V2_Weights

warnings.filterwarnings('ignore')

# ── Device setup ──────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
else:
    print('  → Running on CPU. Training will be slower.')
    print('  → Tip: set NUM_EPOCHS lower for a quick test run.')

---
## ⚙️ Section 2 — Configuration

> **Only edit this cell.** Point `DATA_DIR` to your dataset root.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  ★  EDIT THESE  ★
# ─────────────────────────────────────────────────────────────────────────────

# Root folder containing train/, val/, test/ subfolders
DATA_DIR = r"C:\Users\YourName\Documents\dataset"   # ← CHANGE THIS

# Windows Tesseract path — uncomment and adjust if tesseract is not on PATH
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# ── Training hyperparameters ──────────────────────────────────────────────────
NUM_EPOCHS_CNN  = 10
NUM_EPOCHS_MLP  = 20
BATCH_SIZE      = 32
LEARNING_RATE   = 1e-3
IMG_SIZE        = 224        # MobileNetV2 expects 224x224
RANDOM_SEED     = 42

# ── OCR cache (delete these files to force re-extraction) ────────────────────
OCR_CACHE_DIR   = 'ocr_cache'

# ── Output directories ────────────────────────────────────────────────────────
Path('results').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)
Path(OCR_CACHE_DIR).mkdir(exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Validate DATA_DIR ─────────────────────────────────────────────────────────
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
TRAIN_DIR  = Path(DATA_DIR) / 'train'
VAL_DIR    = Path(DATA_DIR) / 'val'
TEST_DIR   = Path(DATA_DIR) / 'test'

all_ok = True
for split_dir, name in [(TRAIN_DIR, 'train'), (VAL_DIR, 'val'), (TEST_DIR, 'test')]:
    if not split_dir.exists():
        print(f'⚠  {name}/ not found at: {split_dir}')
        all_ok = False
    else:
        classes_found = [d.name for d in sorted(split_dir.iterdir()) if d.is_dir()]
        n_img = sum(1 for p in split_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTS)
        print(f'✅ {name:5s}: {len(classes_found)} classes, {n_img} images  →  {classes_found}')

if all_ok:
    CLASS_NAMES = [d.name for d in sorted(TRAIN_DIR.iterdir()) if d.is_dir()]
    NUM_CLASSES = len(CLASS_NAMES)
    print(f'\nNUM_CLASSES = {NUM_CLASSES}  |  Classes: {CLASS_NAMES}')

---
## 📂 Section 3 — Dataset Loading

We use `torchvision.datasets.ImageFolder` which expects the folder structure:
```
train/
  invoice/   img001.jpg  img002.jpg ...
  receipt/   img003.jpg  img004.jpg ...
  form/      img005.jpg  ...
```

In [ ]:
# ── Image transforms ──────────────────────────────────────────────────────────
# Training: augmentation (flip, rotation, color jitter) to improve generalisation
# Val/Test: only resize + normalize (no randomness)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Datasets ──────────────────────────────────────────────────────────────────
train_dataset = datasets.ImageFolder(str(TRAIN_DIR), transform=train_transform)
val_dataset   = datasets.ImageFolder(str(VAL_DIR),   transform=eval_transform)
test_dataset  = datasets.ImageFolder(str(TEST_DIR),  transform=eval_transform)

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'Train samples : {len(train_dataset)}')
print(f'Val   samples : {len(val_dataset)}')
print(f'Test  samples : {len(test_dataset)}')
print(f'Classes       : {train_dataset.classes}')

# ── Class distribution bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, ds, title in zip(axes,
                          [train_dataset, val_dataset, test_dataset],
                          ['Train', 'Val', 'Test']):
    counts = defaultdict(int)
    for _, label in ds.samples:
        counts[ds.classes[label]] += 1
    ax.bar(counts.keys(), counts.values(), color='#4C9BE8', alpha=0.85, edgecolor='white')
    ax.set_title(f'{title} Distribution', fontsize=12)
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    ax.spines[['top','right']].set_visible(False)
plt.suptitle('Class Distribution Across Splits', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Sample image grid ─────────────────────────────────────────────────────────
un_mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
un_std  = torch.tensor(IMAGENET_STD).view(3,1,1)

sample_imgs, sample_labels = next(iter(train_loader))
n_show = min(8, len(sample_imgs))
fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
for i, ax in enumerate(axes):
    img = (sample_imgs[i] * un_std + un_mean).clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(train_dataset.classes[sample_labels[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images (after augmentation)', fontsize=12)
plt.tight_layout()
plt.savefig('results/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🧠 Section 4 — CNN Pipeline (MobileNetV2)

We use **MobileNetV2** — a lightweight architecture designed for efficient image classification. We replace its final classifier layer to match our number of document classes.

**Strategy:** Fine-tune the full network (not frozen) since document images differ significantly from ImageNet.

In [ ]:
# ── Build MobileNetV2 ─────────────────────────────────────────────────────────
def build_cnn(num_classes: int) -> nn.Module:
    """
    MobileNetV2 pretrained on ImageNet.
    Last classifier layer replaced for our num_classes.
    """
    model = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes),
    )
    return model


cnn_model = build_cnn(NUM_CLASSES).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in cnn_model.parameters())
trainable    = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f'MobileNetV2 loaded')
print(f'  Total params    : {total_params:,}')
print(f'  Trainable params: {trainable:,}')
print(f'  Device          : {DEVICE}')

In [ ]:
# ── Loss, Optimizer, Scheduler ────────────────────────────────────────────────
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
cnn_scheduler = optim.lr_scheduler.StepLR(cnn_optimizer, step_size=4, gamma=0.5)


# ── Training & validation functions ───────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            preds       = outputs.argmax(1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


# ── CNN Training loop ─────────────────────────────────────────────────────────
print(f'Training CNN for {NUM_EPOCHS_CNN} epochs…\n')
cnn_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_cnn_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS_CNN + 1):
    tr_loss, tr_acc = train_one_epoch(
        cnn_model, train_loader, cnn_criterion, cnn_optimizer, DEVICE)
    vl_loss, vl_acc, _, _ = evaluate(
        cnn_model, val_loader, cnn_criterion, DEVICE)
    cnn_scheduler.step()

    cnn_history['train_loss'].append(tr_loss)
    cnn_history['val_loss'].append(vl_loss)
    cnn_history['train_acc'].append(tr_acc)
    cnn_history['val_acc'].append(vl_acc)

    if vl_acc > best_cnn_val_acc:
        best_cnn_val_acc = vl_acc
        torch.save(cnn_model.state_dict(), 'models/best_cnn.pth')
        saved_marker = ' ← saved'
    else:
        saved_marker = ''

    print(f'Epoch [{epoch:02d}/{NUM_EPOCHS_CNN}]  '
          f'Train Loss: {tr_loss:.4f}  Train Acc: {tr_acc:.4f}  '
          f'Val Loss: {vl_loss:.4f}  Val Acc: {vl_acc:.4f}{saved_marker}')

print(f'\n✅ Best Val Accuracy (CNN): {best_cnn_val_acc:.4f}')

In [ ]:
# ── CNN Training curves ───────────────────────────────────────────────────────
epochs_x = range(1, NUM_EPOCHS_CNN + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_x, cnn_history['train_loss'], 'o-', label='Train', color='#4C9BE8')
ax1.plot(epochs_x, cnn_history['val_loss'],   's--', label='Val',  color='#F4845F')
ax1.set_title('CNN — Loss', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.spines[['top','right']].set_visible(False)

ax2.plot(epochs_x, cnn_history['train_acc'], 'o-', label='Train', color='#4C9BE8')
ax2.plot(epochs_x, cnn_history['val_acc'],   's--', label='Val',  color='#F4845F')
ax2.set_title('CNN — Accuracy', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.05); ax2.legend()
ax2.spines[['top','right']].set_visible(False)

plt.suptitle('MobileNetV2 Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('results/cnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Final test evaluation ─────────────────────────────────────────────────────
cnn_model.load_state_dict(torch.load('models/best_cnn.pth', map_location=DEVICE))
_, cnn_test_acc, cnn_preds, cnn_true = evaluate(
    cnn_model, test_loader, cnn_criterion, DEVICE)
print(f'CNN Test Accuracy: {cnn_test_acc:.4f}')

---
## 🔍 Section 5 — OCR Text Extraction

We use **Tesseract OCR** to convert document images into raw text. The extracted text is then cleaned and fed into a TF-IDF vectorizer before being passed to our PyTorch MLP.

> Results are cached to disk — delete `ocr_cache/` to force re-extraction.

In [ ]:
# ── Verify Tesseract ──────────────────────────────────────────────────────────
import shutil as _shutil
tess_path = _shutil.which('tesseract')
if tess_path:
    print(f'✅ Tesseract found at: {tess_path}')
else:
    print('⚠  Tesseract not found on PATH.')
    print('   Windows fix: uncomment tesseract_cmd line in Section 2.')
try:
    ver = pytesseract.get_tesseract_version()
    print(f'   Version: {ver}')
except Exception as e:
    print(f'   Could not get version: {e}')

In [ ]:
# ── Preprocessing for OCR ─────────────────────────────────────────────────────
TESS_CONFIG = '--oem 3 --psm 6 -l eng'


def preprocess_for_ocr(image_path: str) -> np.ndarray:
    """Upscale → Grayscale → Denoise → Adaptive Threshold."""
    img = cv2.imread(str(image_path))
    if img is None:
        pil = Image.open(str(image_path)).convert('RGB')
        img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    h, w = img.shape[:2]
    if max(h, w) < 1000:
        scale = 1000 / max(h, w)
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10,
                                         templateWindowSize=7, searchWindowSize=21)
    binary   = cv2.adaptiveThreshold(denoised, 255,
                                     cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, blockSize=31, C=10)
    return binary


def extract_text(image_path: str) -> str:
    """Run Tesseract and return cleaned text."""
    try:
        img  = preprocess_for_ocr(image_path)
        text = pytesseract.image_to_string(img, config=TESS_CONFIG)
        return ' '.join(text.split()).strip()
    except Exception:
        return ''


def clean_text(text: str) -> str:
    """Lowercase, remove single-char artifacts, keep alphanumeric."""
    text = str(text).lower()
    text = re.sub(r'\b\w{1}\b', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def extract_split(
    split_dir: Path, cache_file: str, desc: str = 'OCR'
) -> pd.DataFrame:
    """Walk split dir, run OCR on each image, cache results."""
    cache_path = Path(OCR_CACHE_DIR) / cache_file
    if cache_path.exists():
        print(f'  Loading from cache: {cache_path}  (delete to re-run)')
        return pd.read_json(cache_path)

    records = []
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    for class_dir in class_dirs:
        paths = [p for p in class_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTS]
        for p in tqdm(paths, desc=f'{desc}/{class_dir.name}', leave=False):
            raw   = extract_text(str(p))
            clean = clean_text(raw)
            records.append({
                'path': str(p), 'label': class_dir.name,
                'raw_text': raw, 'clean_text': clean,
                'word_count': len(clean.split())
            })

    df = pd.DataFrame(records)
    df.to_json(cache_path, orient='records', indent=2)
    print(f'  Saved {len(df)} records → {cache_path}')
    return df


# ── Extract all splits ────────────────────────────────────────────────────────
print('Extracting OCR text…  (this may take a while)\n')
df_train_ocr = extract_split(TRAIN_DIR, 'ocr_train.json', 'Train')
df_val_ocr   = extract_split(VAL_DIR,   'ocr_val.json',   'Val')
df_test_ocr  = extract_split(TEST_DIR,  'ocr_test.json',  'Test')

print()
for name, df in [('Train', df_train_ocr), ('Val', df_val_ocr), ('Test', df_test_ocr)]:
    empty_pct = (df['word_count'] < 3).mean()
    print(f'{name:5s}: {len(df):>5} samples  avg_words: {df["word_count"].mean():.0f}'
          f'  empty OCR: {empty_pct:.1%}')

# OCR quality warning
if (df_train_ocr['word_count'] < 3).mean() > 0.5:
    print('\n⚠  More than 50% of training images have fewer than 3 OCR words.')
    print('   Check Tesseract setup and delete ocr_cache/ to re-extract.')

---
## 🤖 Section 6 — OCR-MLP Pipeline (PyTorch)

Pipeline:
1. TF-IDF vectorize the OCR text (sklearn — feature extraction only)
2. Convert TF-IDF matrix → PyTorch tensors
3. Train a **PyTorch MLP** (3-layer feed-forward network)

In [ ]:
# ── TF-IDF vectorization ──────────────────────────────────────────────────────
MIN_WORDS = 3

# Filter out near-empty OCR results
df_tr = df_train_ocr[df_train_ocr['word_count'] >= MIN_WORDS].copy()
df_v  = df_val_ocr[df_val_ocr['word_count']   >= MIN_WORDS].copy()
df_te = df_test_ocr[df_test_ocr['word_count'] >= MIN_WORDS].copy()

if df_tr.empty:
    raise RuntimeError(
        'No training samples with OCR text found.\n'
        'Fix Tesseract, delete ocr_cache/, and re-run Section 5.'
    )

# Label encoding
le = LabelEncoder()
le.fit(df_tr['label'])
ocr_classes = le.classes_

# Keep only classes seen in training
df_v  = df_v[df_v['label'].isin(ocr_classes)].copy()
df_te = df_te[df_te['label'].isin(ocr_classes)].copy()

# TF-IDF (unigrams + bigrams)
tfidf = TfidfVectorizer(
    sublinear_tf=True, ngram_range=(1, 2),
    max_features=30_000, min_df=2,
    strip_accents='unicode', token_pattern=r'(?u)\b\w+\b'
)
X_tr_raw = tfidf.fit_transform(df_tr['clean_text']).toarray().astype(np.float32)
X_v_raw  = tfidf.transform(df_v['clean_text']).toarray().astype(np.float32)
X_te_raw = tfidf.transform(df_te['clean_text']).toarray().astype(np.float32)

y_tr = le.transform(df_tr['label'])
y_v  = le.transform(df_v['label'])
y_te = le.transform(df_te['label'])

TFIDF_DIM = X_tr_raw.shape[1]
print(f'TF-IDF feature dim : {TFIDF_DIM}')
print(f'Train samples      : {len(X_tr_raw)}')
print(f'Val   samples      : {len(X_v_raw)}')
print(f'Test  samples      : {len(X_te_raw)}')

In [ ]:
# ── PyTorch Dataset & DataLoaders ─────────────────────────────────────────────
class TextDataset(Dataset):
    """Wraps TF-IDF feature matrix and labels as a PyTorch Dataset."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


text_train_loader = DataLoader(
    TextDataset(X_tr_raw, y_tr), batch_size=64, shuffle=True)
text_val_loader   = DataLoader(
    TextDataset(X_v_raw,  y_v),  batch_size=64, shuffle=False)
text_test_loader  = DataLoader(
    TextDataset(X_te_raw, y_te), batch_size=64, shuffle=False)


# ── MLP model ─────────────────────────────────────────────────────────────────
class DocumentMLP(nn.Module):
    """
    3-layer MLP for TF-IDF document classification.
    input_dim  → 1024 → 256 → num_classes
    """
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.net(x)


mlp_model = DocumentMLP(TFIDF_DIM, NUM_CLASSES).to(DEVICE)
total_mlp = sum(p.numel() for p in mlp_model.parameters())
print(f'MLP loaded  |  Total params: {total_mlp:,}  |  Input dim: {TFIDF_DIM}')

In [ ]:
# ── MLP Training ──────────────────────────────────────────────────────────────
mlp_criterion = nn.CrossEntropyLoss()
mlp_optimizer = optim.Adam(mlp_model.parameters(), lr=1e-3, weight_decay=1e-4)
mlp_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    mlp_optimizer, mode='max', patience=3, factor=0.5)

print(f'Training MLP for {NUM_EPOCHS_MLP} epochs…\n')
mlp_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_mlp_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS_MLP + 1):
    tr_loss, tr_acc = train_one_epoch(
        mlp_model, text_train_loader, mlp_criterion, mlp_optimizer, DEVICE)
    vl_loss, vl_acc, _, _ = evaluate(
        mlp_model, text_val_loader, mlp_criterion, DEVICE)
    mlp_scheduler.step(vl_acc)

    mlp_history['train_loss'].append(tr_loss)
    mlp_history['val_loss'].append(vl_loss)
    mlp_history['train_acc'].append(tr_acc)
    mlp_history['val_acc'].append(vl_acc)

    if vl_acc > best_mlp_val_acc:
        best_mlp_val_acc = vl_acc
        torch.save(mlp_model.state_dict(), 'models/best_mlp.pth')
        saved_marker = ' ← saved'
    else:
        saved_marker = ''

    print(f'Epoch [{epoch:02d}/{NUM_EPOCHS_MLP}]  '
          f'Train Loss: {tr_loss:.4f}  Train Acc: {tr_acc:.4f}  '
          f'Val Loss: {vl_loss:.4f}  Val Acc: {vl_acc:.4f}{saved_marker}')

print(f'\n✅ Best Val Accuracy (MLP): {best_mlp_val_acc:.4f}')

# ── MLP training curves ───────────────────────────────────────────────────────
epochs_x = range(1, NUM_EPOCHS_MLP + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_x, mlp_history['train_loss'], 'o-', label='Train', color='#9B59B6')
ax1.plot(epochs_x, mlp_history['val_loss'],   's--', label='Val',  color='#2ECC71')
ax1.set_title('OCR-MLP — Loss', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.spines[['top','right']].set_visible(False)

ax2.plot(epochs_x, mlp_history['train_acc'], 'o-', label='Train', color='#9B59B6')
ax2.plot(epochs_x, mlp_history['val_acc'],   's--', label='Val',  color='#2ECC71')
ax2.set_title('OCR-MLP — Accuracy', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.05); ax2.legend()
ax2.spines[['top','right']].set_visible(False)

plt.suptitle('OCR-MLP Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('results/mlp_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Final test evaluation ─────────────────────────────────────────────────────
mlp_model.load_state_dict(torch.load('models/best_mlp.pth', map_location=DEVICE))
_, mlp_test_acc, mlp_preds, mlp_true = evaluate(
    mlp_model, text_test_loader, mlp_criterion, DEVICE)
print(f'MLP Test Accuracy: {mlp_test_acc:.4f}')

---
## 📊 Section 7 — Evaluation & Comparison

Side-by-side comparison of CNN and OCR-MLP on the test set.

In [ ]:
# ── Accuracy comparison table ─────────────────────────────────────────────────
summary = pd.DataFrame({
    'Model': ['MobileNetV2 (CNN)', 'OCR-MLP (PyTorch)'],
    'Best Val Acc': [best_cnn_val_acc, best_mlp_val_acc],
    'Test Acc':     [cnn_test_acc,     mlp_test_acc],
    'Input':        ['Image pixels',   'TF-IDF from OCR'],
    'DL Framework': ['PyTorch',        'PyTorch'],
})
print('=' * 70)
print(summary.to_string(index=False))
print('=' * 70)
summary.to_csv('results/model_comparison.csv', index=False)

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(2)
w = 0.35
b1 = ax.bar(x - w/2, summary['Best Val Acc'], w,
            label='Val Acc', color='#4C9BE8', alpha=0.88)
b2 = ax.bar(x + w/2, summary['Test Acc'], w,
            label='Test Acc', color='#F4845F', alpha=0.88)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(summary['Model'], fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Accuracy')
ax.set_title('CNN vs OCR-MLP — Accuracy Comparison', fontsize=14)
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    cm   = confusion_matrix(y_true, y_pred)
    norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    n    = len(class_names)
    fig, ax = plt.subplots(figsize=(max(8, n*0.8), max(6, n*0.7)))
    sns.heatmap(norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.4, vmin=0, vmax=1, ax=ax,
                cbar_kws={'label': 'Recall'})
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.xticks(rotation=35, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {save_path}')


plot_confusion_matrix(
    cnn_true, cnn_preds, CLASS_NAMES,
    'CNN (MobileNetV2) — Confusion Matrix (Test)',
    'results/cm_cnn.png'
)

plot_confusion_matrix(
    mlp_true, mlp_preds, list(ocr_classes),
    'OCR-MLP — Confusion Matrix (Test)',
    'results/cm_mlp.png'
)

In [ ]:
# ── Classification reports ────────────────────────────────────────────────────
print('=' * 55)
print('  CNN (MobileNetV2) — Classification Report')
print('=' * 55)
print(classification_report(cnn_true, cnn_preds, target_names=CLASS_NAMES))

print('=' * 55)
print('  OCR-MLP — Classification Report')
print('=' * 55)
print(classification_report(mlp_true, mlp_preds, target_names=list(ocr_classes)))

# Save to file
with open('results/classification_reports.txt', 'w') as f:
    f.write('CNN (MobileNetV2)\n')
    f.write(classification_report(cnn_true, cnn_preds, target_names=CLASS_NAMES))
    f.write('\nOCR-MLP\n')
    f.write(classification_report(mlp_true, mlp_preds, target_names=list(ocr_classes)))
print('Saved → results/classification_reports.txt')

---
## 🔄 Section 8 — Robustness Testing

We test both models on **three perturbations** simulating real-world conditions:
- **Rotation** (5°, 15°, 30°, 90°)
- **Salt-and-pepper noise** (1%, 5%, 10%)
- **Gaussian blur** (kernel sizes 3, 5, 9)

A performance drop chart compares CNN vs OCR-MLP robustness.

In [ ]:
# ── Augmentation functions ────────────────────────────────────────────────────
def rotate_image(img: np.ndarray, angle: float) -> np.ndarray:
    h, w = img.shape[:2]
    M    = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    cos, sin = abs(M[0,0]), abs(M[0,1])
    new_w = int(h*sin + w*cos)
    new_h = int(h*cos + w*sin)
    M[0,2] += (new_w - w)/2
    M[1,2] += (new_h - h)/2
    return cv2.warpAffine(img, M, (new_w, new_h),
                          flags=cv2.INTER_CUBIC, borderValue=255)


def add_salt_pepper(img: np.ndarray, amount: float = 0.02) -> np.ndarray:
    noisy = img.copy()
    n     = int(amount * img.size)
    r1    = np.random.randint(0, img.shape[0], n)
    c1    = np.random.randint(0, img.shape[1], n)
    noisy[r1, c1] = 255
    r2    = np.random.randint(0, img.shape[0], n)
    c2    = np.random.randint(0, img.shape[1], n)
    noisy[r2, c2] = 0
    return noisy


def add_blur(img: np.ndarray, ksize: int = 5) -> np.ndarray:
    return cv2.GaussianBlur(img, (ksize, ksize), 0)


# ── Helpers: perturb → evaluate CNN ──────────────────────────────────────────
def perturb_and_eval_cnn(perturb_fn, num_samples: int = 30) -> float:
    """Sample test images, apply perturbation, evaluate CNN."""
    all_preds, all_true = [], []
    sample_paths = [(p, l) for p, l in test_dataset.samples]
    random.shuffle(sample_paths)
    sample_paths = sample_paths[:num_samples]

    tmp_path = tempfile.mktemp(suffix='.png')
    for path, label in sample_paths:
        raw = cv2.imread(str(path))
        if raw is None:
            continue
        aug = perturb_fn(raw)
        cv2.imwrite(tmp_path, aug)
        try:
            img_t = eval_transform(Image.open(tmp_path).convert('RGB')).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                pred = cnn_model(img_t).argmax(1).item()
            all_preds.append(pred)
            all_true.append(label)
        except Exception:
            pass
    try:
        os.remove(tmp_path)
    except Exception:
        pass
    return accuracy_score(all_true, all_preds) if all_preds else 0.0


def perturb_and_eval_mlp(perturb_fn, num_samples: int = 30) -> float:
    """Sample test images, apply perturbation, run OCR, evaluate MLP."""
    all_preds, all_true = [], []
    sample_records = list(zip(df_test_ocr['path'], df_test_ocr['label']))
    random.shuffle(sample_records)
    sample_records = [(p, l) for p, l in sample_records if l in set(ocr_classes)]
    sample_records = sample_records[:num_samples]

    tmp_path = tempfile.mktemp(suffix='.png')
    for path, label in sample_records:
        raw = cv2.imread(str(path))
        if raw is None:
            continue
        aug = perturb_fn(raw)
        cv2.imwrite(tmp_path, aug)
        try:
            text   = clean_text(extract_text(tmp_path))
            feat   = tfidf.transform([text]).toarray().astype(np.float32)
            feat_t = torch.from_numpy(feat).to(DEVICE)
            with torch.no_grad():
                pred = mlp_model(feat_t).argmax(1).item()
            all_preds.append(pred)
            all_true.append(le.transform([label])[0])
        except Exception:
            pass
    try:
        os.remove(tmp_path)
    except Exception:
        pass
    return accuracy_score(all_true, all_preds) if all_preds else 0.0


print('Robustness evaluation… (OCR perturbations take longer)\n')
ROB_SAMPLES = 20   # increase for more reliable estimates

experiments = [
    ('Rotation  5°',      lambda img: rotate_image(img, 5)),
    ('Rotation 15°',      lambda img: rotate_image(img, 15)),
    ('Rotation 30°',      lambda img: rotate_image(img, 30)),
    ('Rotation 90°',      lambda img: rotate_image(img, 90)),
    ('Noise  1%',         lambda img: add_salt_pepper(img, 0.01)),
    ('Noise  5%',         lambda img: add_salt_pepper(img, 0.05)),
    ('Noise 10%',         lambda img: add_salt_pepper(img, 0.10)),
    ('Blur k=3',          lambda img: add_blur(img, 3)),
    ('Blur k=5',          lambda img: add_blur(img, 5)),
    ('Blur k=9',          lambda img: add_blur(img, 9)),
]

# Baseline (no perturbation)
cnn_baseline = cnn_test_acc
mlp_baseline = mlp_test_acc

rob_results = []
for exp_name, fn in tqdm(experiments, desc='Perturbations'):
    cnn_rob = perturb_and_eval_cnn(fn, ROB_SAMPLES)
    mlp_rob = perturb_and_eval_mlp(fn, ROB_SAMPLES)
    rob_results.append({
        'Perturbation': exp_name,
        'CNN Acc':      cnn_rob,
        'CNN Drop':     cnn_baseline - cnn_rob,
        'MLP Acc':      mlp_rob,
        'MLP Drop':     mlp_baseline - mlp_rob,
    })
    print(f'  {exp_name:<15}  CNN: {cnn_rob:.4f} (Δ{cnn_baseline-cnn_rob:+.4f})  '
          f'MLP: {mlp_rob:.4f} (Δ{mlp_baseline-mlp_rob:+.4f})')

rob_df = pd.DataFrame(rob_results)
rob_df.to_csv('results/robustness_results.csv', index=False)
print('\nSaved → results/robustness_results.csv')

In [ ]:
# ── Robustness plot ───────────────────────────────────────────────────────────
x_labels = rob_df['Perturbation'].tolist()
x        = np.arange(len(x_labels))
w        = 0.35

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Top: absolute accuracy
ax1.plot(x, [cnn_baseline]*len(x), '--', color='#4C9BE8', alpha=0.5, label='CNN Baseline')
ax1.plot(x, [mlp_baseline]*len(x), '--', color='#9B59B6', alpha=0.5, label='MLP Baseline')
ax1.bar(x - w/2, rob_df['CNN Acc'], w, label='CNN', color='#4C9BE8', alpha=0.85)
ax1.bar(x + w/2, rob_df['MLP Acc'], w, label='MLP', color='#9B59B6', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(x_labels, rotation=35, ha='right', fontsize=9)
ax1.set_ylabel('Accuracy'); ax1.set_ylim(0, 1.1)
ax1.set_title('Robustness: Accuracy under Perturbation', fontsize=13)
ax1.legend(); ax1.spines[['top','right']].set_visible(False)

# Bottom: accuracy drop
ax2.bar(x - w/2, rob_df['CNN Drop'], w, label='CNN Drop', color='#F4845F', alpha=0.85)
ax2.bar(x + w/2, rob_df['MLP Drop'], w, label='MLP Drop', color='#2ECC71', alpha=0.85)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(x_labels, rotation=35, ha='right', fontsize=9)
ax2.set_ylabel('Accuracy Drop (vs baseline)')
ax2.set_title('Robustness: Accuracy Drop', fontsize=13)
ax2.legend(); ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/robustness.png')

---
## 🔀 Section 9 — Fusion Model

**Idea:** Combine the CNN and OCR-MLP probability outputs using **weighted averaging**.  
The weight `alpha` controls how much we trust the CNN vs the MLP:

```
fusion_prob = alpha × CNN_prob + (1 - alpha) × MLP_prob
```

We grid-search `alpha ∈ [0.0, 0.1, ..., 1.0]` on the **val set** and report test accuracy.

In [ ]:
# ── Collect softmax probabilities from val set ─────────────────────────────────
def get_probs_and_labels(
    model: nn.Module, loader: DataLoader, device: torch.device
):
    """Returns (probs [N, C], labels [N]) as numpy arrays."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            probs = F.softmax(model(X), dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.numpy())
    return np.vstack(all_probs), np.concatenate(all_labels)


# CNN val probs — only for images where we also have OCR
# For simplicity we use the full val loaders independently
cnn_val_probs, cnn_val_true  = get_probs_and_labels(cnn_model, val_loader, DEVICE)
mlp_val_probs, mlp_val_true  = get_probs_and_labels(mlp_model, text_val_loader, DEVICE)
cnn_test_probs, cnn_test_true = get_probs_and_labels(cnn_model, test_loader, DEVICE)
mlp_test_probs, mlp_test_true = get_probs_and_labels(mlp_model, text_test_loader, DEVICE)

print(f'CNN val probs shape : {cnn_val_probs.shape}')
print(f'MLP val probs shape : {mlp_val_probs.shape}')

# ── Grid-search alpha on val set ─────────────────────────────────────────────
# NOTE: Fusion requires both models to have the same number of val samples.
# If sizes differ (due to OCR filtering), we align to the smaller set.
n_val = min(len(cnn_val_probs), len(mlp_val_probs))
n_te  = min(len(cnn_test_probs), len(mlp_test_probs))

cnn_v_p = cnn_val_probs[:n_val]
mlp_v_p = mlp_val_probs[:n_val]
cnn_v_t = cnn_val_true[:n_val]
mlp_v_t = mlp_val_true[:n_val]

# Use CNN true labels (both models share same class ordering)
alphas     = np.arange(0.0, 1.01, 0.1)
val_accs   = []
best_alpha = 0.5
best_val   = 0.0

for alpha in alphas:
    fused = alpha * cnn_v_p + (1 - alpha) * mlp_v_p
    preds = fused.argmax(axis=1)
    acc   = accuracy_score(cnn_v_t, preds)
    val_accs.append(acc)
    if acc > best_val:
        best_val   = acc
        best_alpha = alpha

print(f'\nBest alpha = {best_alpha:.1f}  (Val Acc = {best_val:.4f})')

# ── Alpha sweep plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(alphas, val_accs, 'o-', color='#E74C3C', linewidth=2)
ax.axvline(best_alpha, color='gray', linestyle='--', alpha=0.7,
           label=f'Best α={best_alpha:.1f}')
ax.axhline(cnn_test_acc, color='#4C9BE8', linestyle=':', label=f'CNN only ({cnn_test_acc:.3f})')
ax.axhline(mlp_test_acc, color='#9B59B6', linestyle=':', label=f'MLP only ({mlp_test_acc:.3f})')
ax.set_xlabel('Alpha (weight for CNN)', fontsize=11)
ax.set_ylabel('Val Accuracy', fontsize=11)
ax.set_title('Fusion Alpha Sweep (α × CNN + (1-α) × MLP)', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.1)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/fusion_alpha_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Evaluate fusion on test set ───────────────────────────────────────────────
cnn_te_p = cnn_test_probs[:n_te]
mlp_te_p = mlp_test_probs[:n_te]
cnn_te_t = cnn_test_true[:n_te]

fused_test      = best_alpha * cnn_te_p + (1 - best_alpha) * mlp_te_p
fusion_preds    = fused_test.argmax(axis=1)
fusion_test_acc = accuracy_score(cnn_te_t, fusion_preds)

print(f'\n{"="*50}')
print(f'  FINAL TEST RESULTS')
print(f'{"="*50}')
print(f'  CNN (MobileNetV2) : {cnn_test_acc:.4f}')
print(f'  OCR-MLP           : {mlp_test_acc:.4f}')
print(f'  Fusion (α={best_alpha:.1f})   : {fusion_test_acc:.4f}')
print(f'{"="*50}')

# Save final results
final = {
    'CNN Test Acc':    cnn_test_acc,
    'MLP Test Acc':    mlp_test_acc,
    'Fusion Test Acc': fusion_test_acc,
    'Best Alpha':      best_alpha,
}
with open('results/final_results.json', 'w') as f:
    json.dump(final, f, indent=2)
print('\nSaved → results/final_results.json')

In [ ]:
# ── Final confusion matrix for fusion ─────────────────────────────────────────
plot_confusion_matrix(
    cnn_te_t, fusion_preds, CLASS_NAMES,
    f'Fusion Model (α={best_alpha:.1f}) — Confusion Matrix (Test)',
    'results/cm_fusion.png'
)

# ── Final summary bar chart ───────────────────────────────────────────────────
models_  = ['CNN\n(MobileNetV2)', 'OCR-MLP\n(PyTorch)', f'Fusion\n(α={best_alpha:.1f})']
accs_    = [cnn_test_acc, mlp_test_acc, fusion_test_acc]
colors_  = ['#4C9BE8', '#9B59B6', '#E74C3C']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models_, accs_, color=colors_, alpha=0.88, edgecolor='white', width=0.5)
for bar, acc in zip(bars, accs_):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.012,
            f'{acc:.4f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Final Model Comparison — Test Accuracy', fontsize=14)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Notebook complete. All outputs saved in results/ and models/')
print('\nOutput files:')
for p in sorted(Path('results').glob('*')):
    print(f'  results/{p.name}')
for p in sorted(Path('models').glob('*')):
    print(f'  models/{p.name}')